In [8]:
"""
- 输入：把 30x30 迷宫展平成长度 900 的一维向量（每个格子映射到 0~4）
- 网络：900 -> 4096 -> 512 -> 4（ReLU）
- 训练：用 train_data.csv + train_answer.csv 做回归（MSE），Adam，小批量（PyTorch）
- 输出：对 test_data.csv 预测，写 result.csv（每行 4 个实数）

用法：
    python baseline.py train_data.csv train_answer.csv test_data.csv result.csv
"""

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import zipfile
import os
import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

N = 30
D = N * N

In [9]:
# ===== 数据读取与编码 =====

def encode_lines(path: str) -> np.ndarray:
    # 将每行 900 字符编码成 0~4 的向量：
    # '.'->0, '# '->1, '?'->2, 'S'->3, 'T'->4
    char_id = {".": 0, "#": 1, "?": 2, "S": 3, "T": 4}
    with open(path, "r", encoding="utf-8-sig") as f:
        xs = [[char_id[c] for c in line.strip()] for line in f if line.strip()]
    return np.asarray(xs, dtype=np.long).reshape(-1, 30, 30)


def read_y(path: str) -> np.ndarray:
    # 读取 4 列标签（整数），训练时当作 float
    return np.loadtxt(path, delimiter=",", dtype=np.float32, encoding="utf-8-sig")

In [10]:
# ===== 结果写出 =====

def write_result(path: Path, pred: np.ndarray) -> None:
    np.savetxt(path, pred, delimiter=",", fmt="%.6f")

In [14]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=5, embedding_dim=4):
        super(CNNModel, self).__init__()
        self.embedding = nn.Embedding(num_classes, embedding_dim)
        self.conv1 = nn.Conv2d(embedding_dim, 8, kernel_size=5, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        self.maxpool2d = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(32 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, 4)

    def forward(self, x):
        x = self.embedding(x)  # (B, 30, 30) -> (B, 30, 30, 8)
        x = x.permute(0, 3, 1, 2)
        x = self.bn1(self.conv1(x))
        x = self.maxpool2d(self.relu(x))
        x = self.bn2(self.conv2(x))
        x = self.maxpool2d(self.relu(x))
        x = self.bn3(self.conv3(x))
        x = self.maxpool2d(self.relu(x))
        x = x.reshape(x.size(0), -1)  # Flatten
        x = self.fc1(x)  # (B, 4)
        x = self.relu(x)
        x = self.fc2(x)  # (B, 4)
        return x

def train_model(train_x_path: str, train_y_path: str, epochs: int) -> nn.Module:
    x_train = torch.from_numpy(encode_lines(train_x_path)).long()  # Ensure x_train is of type long for embedding
    y_train = torch.from_numpy(read_y(train_y_path)).float()
    mean = y_train.mean(dim=0)
    std = y_train.std(dim=0)
    y_train_normalized = (y_train - mean) / std

    torch.manual_seed(0)
    model = CNNModel()
    loader = DataLoader(TensorDataset(x_train, y_train_normalized), batch_size=64, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

    model.train()
    for ep in range(1, epochs + 1):
        total = 0.0
        for xb, yb in loader:
            pred = model(xb)
            loss = nn.functional.mse_loss(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss.item()) * xb.shape[0]
        print(f"epoch {ep}/{epochs}  mse={total / len(x_train):.6f}")
    return model, mean, std

In [15]:
# ===== 预测 =====

def predict(model: nn.Module, test_x_path: str, mean, std) -> np.ndarray:
    x_test = torch.from_numpy(encode_lines(test_x_path))
    model.eval()
    with torch.no_grad():
        return (model(x_test) * std + mean).cpu().numpy()


In [16]:


# ===== 主流程 =====


TRAIN_PATH = "/bohr/train-abk9/v1/"  # 训练集路径


# 训练集
train_x_path = TRAIN_PATH + "train_data.csv"
train_y_path = TRAIN_PATH + "train_answer.csv"



out_path = Path("result.csv")
epochs = 8

model, mean, std = train_model(train_x_path, train_y_path, epochs)




# 保存模型权重和必要的缩放信息，便于复现预测。
torch.save(
    {
        "state_dict": model.state_dict(),
        "epochs": epochs,
        "architecture": "900-4096-512-4",
        "input_scale": 4.0,
        "output_scale": 900.0,
    },
    out_path.with_name("model.pt"),
)


In [7]:
if os.environ.get("DATA_PATH"):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  # 测试集路径
else:
    DATA_PATH = "/bohr/mazeval-7zx2/v1/"  # 本地测试回退

# 测试集
testA_path = DATA_PATH + "val_data.csv"
testB_path = DATA_PATH + "test_data.csv"
# testA_path = "/bohr/train-abk9/v1/train_data.csv"
# testB_path = "/bohr/train-abk9/v1/train_data.csv"

#分别预测
pred_A = predict(model, testA_path, mean, std)
pred_B = predict(model, testB_path, mean, std)

#合并预测结果

submissionA = pd.DataFrame(pred_A)
submissionA.to_csv("./submission_val.csv", index=False, header=False)

submissionB = pd.DataFrame(pred_B)
submissionB.to_csv("./submission_test.csv", index=False, header=False)

files_to_zip = ['./submission_val.csv', './submission_test.csv']
zip_filename = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')